# pronto vou começar a parte 3

# 1. Are there highly correlated features?
Queria perceber se há variáveis audio que medem praticamente a mesma coisa, ajudando a identificar redundancia e preparar para a next question redução dos dados é possivel

In [ ]:
# começamos por excolher as features do audio (vi na parte 1 quais as features que os ficheiros audio tinham)

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path(".")
audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

all_dfs = []

for file_path in audio_files:
    df = pd.read_pickle(file_path).copy()
    df["source_file"] = file_path.name
    all_dfs.append(df)

audio_clean = pd.concat(all_dfs, ignore_index=True)

print("Dimensão:", audio_clean.shape)

audio_features = [
    "duration",
    "meanF0Hz",
    "stdevF0Hz",
    "HNR",
    "localJitter",
    "localabsoluteJitter",
    "rapJitter",
    "ppq5Jitter",
    "ddpJitter",
    "localShimmer",
    "localdbShimmer",
    "apq3Shimmer",
    "apq5Shimmer",
    "apq11Shimmer",
    "ddaShimmer",
    "npause",
    "speechrate",
    "articulationrate",
    "asd",
    "pF",
    "fdisp",
    "avgFormant",
    "mff"
]

corr_data = audio_clean[audio_features].copy()
corr_matrix = corr_data.corr()

corr_matrix.round(2)

In [ ]:
# pronto, agora fazer o heatmap para entender como se relacionam 

plt.figure(figsize=(12, 10))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar(label="Correlation")

plt.xticks(range(len(audio_features)), audio_features, rotation=90)
plt.yticks(range(len(audio_features)), audio_features)

plt.title("Correlation matrix between audio features")
plt.tight_layout()
plt.show()

In [ ]:
# quais são as features de audio que mais se correlacionam??
# a ideia seria: calcular a correlação entre features áudio para perceber se algumas variáveis estavam a medir informação parecida. Isto é importante porque, se houver muitas features altamente correlacionadas, há redundância nos dados. Isso justifica depois aplicar PCA para reduzir dimensionalidade.
corr_pairs = (
    corr_matrix
    .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)

corr_pairs.columns = ["feature_1", "feature_2", "correlation"]

strong_corr = corr_pairs[
    corr_pairs["correlation"].abs() >= 0.70
].copy()

strong_corr["abs_correlation"] = strong_corr["correlation"].abs()
strong_corr = strong_corr.sort_values("abs_correlation", ascending=False)

strong_corr

In [ ]:
# transformar a matriz de correlação em pares de features

corr_pairs = (
    corr_matrix
    .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)

corr_pairs.columns = ["feature_1", "feature_2", "correlation"]

corr_pairs["abs_correlation"] = corr_pairs["correlation"].abs()

# escolher as 15 correlações mais fortes
top_corr = corr_pairs.sort_values("abs_correlation", ascending=False).head(15)

top_corr["pair"] = top_corr["feature_1"] + " vs " + top_corr["feature_2"]

top_corr

# correlações acima de 0.7

strong_corr = corr_pairs[corr_pairs["abs_correlation"] >= 0.70].copy()

strong_corr = strong_corr.sort_values("abs_correlation", ascending=False)
strong_corr["pair"] = strong_corr["feature_1"] + " vs " + strong_corr["feature_2"]

plt.figure(figsize=(10, max(5, len(strong_corr) * 0.35)))

plt.barh(strong_corr["pair"], strong_corr["correlation"])

plt.axvline(0, color="black", linewidth=0.8)

plt.title("Highly correlated audio features | correlation >= 0.70")
plt.xlabel("Correlation")
plt.ylabel("Feature pair")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# Features áudio usadas no PCA
audio_features = [
    "duration",
    "meanF0Hz",
    "stdevF0Hz",
    "HNR",
    "localJitter",
    "localabsoluteJitter",
    "rapJitter",
    "ppq5Jitter",
    "ddpJitter",
    "localShimmer",
    "localdbShimmer",
    "apq3Shimmer",
    "apq5Shimmer",
    "apq11Shimmer",
    "ddaShimmer",
    "npause",
    "speechrate",
    "articulationrate",
    "asd",
    "pF",
    "fdisp",
    "avgFormant",
    "mff"
]

# Preparar dados
X = audio_clean[audio_features].copy()

# Normalizar porque as features têm escalas diferentes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA com todas as componentes
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained_variance = pca.explained_variance_ratio_
cumulative_variance = explained_variance.cumsum()

# Plot
plt.figure(figsize=(9, 5))

plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker="o"
)

plt.axhline(0.80, linestyle="--", label="80% variance")
plt.axhline(0.90, linestyle="--", label="90% variance")

plt.title("Cumulative explained variance by PCA components")
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance")
plt.legend()
plt.tight_layout()
plt.show()

# Número de componentes necessárias
n_components_80 = np.argmax(cumulative_variance >= 0.80) + 1
n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1

print("Número de componentes para explicar 80% da variância:", n_components_80)
print("Número de componentes para explicar 90% da variância:", n_components_90)


In [ ]:
# agora com esta informação podemos fazer a next task 
# Can you reduce the dimensionality of the data?
# vamos ver, pelos embeedings e o gráfico se a diferença entre tirar as features correlacionas e não tirar

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from matplotlib.patches import Ellipse

DATA_DIR = Path(".")
audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

debate_file = None

for f in audio_files:
    if "Ventura" in f.name and "Seguro" in f.name:
        debate_file = f
        break

if debate_file is None:
    raise FileNotFoundError("Não encontrei o ficheiro Ventura vs Seguro.")

print("Ficheiro escolhido:", debate_file.name)

df_debate = pd.read_pickle(debate_file).copy()

df_debate["segment_start"] = df_debate["time stamp"]
df_debate["segment_end"] = df_debate["time stamp"] + df_debate["duration"]

embeddings = np.vstack(df_debate["speak_embeddings"].values)
embeddings_norm = normalize(embeddings)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_debate["speaker_cluster"] = kmeans.fit_predict(embeddings_norm)

speaker_time = (
    df_debate
    .groupby("speaker_cluster")
    .agg(total_speech_sec=("duration", "sum"))
    .reset_index()
    .sort_values("total_speech_sec", ascending=False)
)

speaker_time["person"] = [f"Person {i}" for i in range(1, 4)]

cluster_to_person = dict(
    zip(speaker_time["speaker_cluster"], speaker_time["person"])
)

df_debate["person"] = df_debate["speaker_cluster"].map(cluster_to_person)

# Se este mapeamento estiver certo, mantém. Se estiver trocado, troca aqui.
person_to_name = {
    "Person 1": "Ventura",
    "Person 2": "Seguro",
    "Person 3": "Moderador"
}

df_debate["speaker_name"] = df_debate["person"].map(person_to_name)

df_debate[["segment_start", "duration", "person", "speaker_name"]].head()



In [ ]:
# Embeddings completos: 512 dimensões
embeddings_full = embeddings_norm

# Reduzir embeddings: manter só as 100 dimensões com maior variância
embedding_variance = embeddings_full.var(axis=0)

n_selected_features = 100
top_feature_indices = np.argsort(embedding_variance)[-n_selected_features:]

embeddings_reduced = embeddings_full[:, top_feature_indices]

print("Dimensão original:", embeddings_full.shape)
print("Dimensão reduzida:", embeddings_reduced.shape)

# PCA com 512 dimensões
pca_full = PCA(n_components=2)
pca_full_result = pca_full.fit_transform(embeddings_full)

df_debate["PCA1_full"] = pca_full_result[:, 0]
df_debate["PCA2_full"] = pca_full_result[:, 1]

# PCA com 100 dimensões
pca_reduced = PCA(n_components=2)
pca_reduced_result = pca_reduced.fit_transform(embeddings_reduced)

df_debate["PCA1_reduced"] = pca_reduced_result[:, 0]
df_debate["PCA2_reduced"] = pca_reduced_result[:, 1]

print("Variância explicada com 512 dimensões:")
print(pca_full.explained_variance_ratio_)
print("Total:", round(pca_full.explained_variance_ratio_.sum(), 3))

print("\nVariância explicada com 100 dimensões:")
print(pca_reduced.explained_variance_ratio_)
print("Total:", round(pca_reduced.explained_variance_ratio_.sum(), 3))



In [ ]:
# agora dar plot dos dois e ver se a diferença é tão grande aasim 

colors = {
    "Ventura": "tab:blue",
    "Seguro": "tab:orange",
    "Moderador": "tab:green"
}

plt.figure(figsize=(14, 6))

# PCA com embeddings completos
plt.subplot(1, 2, 1)

for speaker in ["Ventura", "Seguro", "Moderador"]:
    data = df_debate[df_debate["speaker_name"] == speaker]
    
    plt.scatter(
        data["PCA1_full"],
        data["PCA2_full"],
        label=speaker,
        alpha=0.7,
        s=45,
        color=colors[speaker]
    )

plt.title("PCA com embeddings completos\n512 dimensões")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.grid(alpha=0.2)
plt.legend()

# PCA com embeddings reduzidos
plt.subplot(1, 2, 2)

for speaker in ["Ventura", "Seguro", "Moderador"]:
    data = df_debate[df_debate["speaker_name"] == speaker]
    
    plt.scatter(
        data["PCA1_reduced"],
        data["PCA2_reduced"],
        label=speaker,
        alpha=0.7,
        s=45,
        color=colors[speaker]
    )

plt.title("PCA com embeddings reduzidos\n100 dimensões")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.grid(alpha=0.2)
plt.legend()

plt.tight_layout()
plt.show()

# para o facto de as dimensões serem cerca de 5x menos não considero que a diferença seja tão grande assim


# passamos agora para a next question
# Can you use some features to predict others?

tentar que algumas features consigam prever outra informação do próprio áudio

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Features áudio usadas para prever quem fala

predictor_features = [
    "duration",
    "meanF0Hz",
    "stdevF0Hz",
    "HNR",
    "localJitter",
    "localShimmer",
    "npause",
    "speechrate",
    "articulationrate",
    "asd",
    "pF",
    "fdisp",
    "avgFormant",
    "mff"
]

# remover linhas sem speaker_name, caso existam
model_data = df_debate.dropna(subset=["speaker_name"]).copy()

X = model_data[predictor_features]
y = model_data["speaker_name"]

print("Número de segmentos:", len(model_data))
print("Distribuição das classes:")
print(y.value_counts())

# dividir dados em treino e teste

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 3))
print("\nClassification report:")
print(classification_report(y_test, y_pred))


# agora tenho de fazer os perfis de cada candidato para ajudar com o visual e dar os dados, 
Queria ter um ficheiro com uma linha por segundo, para todos os debates, com:

quem está a falar (para sabermos, uma vez que já fiz um ficheiro assim, só para ver a pessoa)
se houve troca de speaker
se houve possível interrupção
se houve aumento de speechrate
se houve aumento de variação vocal
se é interessante para comparar com movimento visual.

In [ ]:
# para cada feature audio vou tentar prever usando outras features

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

audio_model = audio_clean.copy()

audio_features = [
    "duration",
    "meanF0Hz",
    "stdevF0Hz",
    "HNR",
    "localJitter",
    "localabsoluteJitter",
    "rapJitter",
    "ppq5Jitter",
    "ddpJitter",
    "localShimmer",
    "localdbShimmer",
    "apq3Shimmer",
    "apq5Shimmer",
    "apq11Shimmer",
    "ddaShimmer",
    "f1_mean",
    "f2_mean",
    "f3_mean",
    "f4_mean",
    "npause",
    "speechrate",
    "articulationrate",
    "asd",
    "pF",
    "fdisp",
    "avgFormant",
    "mff"
]

model_df = audio_model[audio_features + ["source_file"]].copy()
model_df = model_df.replace([np.inf, -np.inf], np.nan).dropna()

print("Dados para modelação:", model_df.shape)



In [ ]:
# testar as features que podem prever outras uma a uma

results = []
importance_rows = []

# separação por debates: treinamos nuns debates e testamos noutros
groups = model_df["source_file"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(splitter.split(model_df, groups=groups))

for target_feature in audio_features:
    
    predictor_features = [
        feature for feature in audio_features
        if feature != target_feature
    ]
    
    X = model_df[predictor_features]
    y = model_df[target_feature]
    
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
    
    model = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=3,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    results.append({
        "target_feature": target_feature,
        "r2_score": r2,
        "mae": mae
    })
    
    for feature, importance in zip(predictor_features, model.feature_importances_):
        importance_rows.append({
            "target_feature": target_feature,
            "predictor_feature": feature,
            "importance": importance
        })

predictability_results = pd.DataFrame(results)
feature_importances = pd.DataFrame(importance_rows)

predictability_results = predictability_results.sort_values(
    "r2_score",
    ascending=False
).reset_index(drop=True)

predictability_results

In [ ]:

# encontrar os pares feature usada para prever e a feauture prevista
top_targets = predictability_results.head(6)["target_feature"].tolist()

top_pairs = []

for target in top_targets:
    temp = (
        feature_importances[
            feature_importances["target_feature"] == target
        ]
        .sort_values("importance", ascending=False)
        .head(3)
        .copy()
    )
    
    temp["pair"] = temp["target_feature"] + "  <-  " + temp["predictor_feature"]
    top_pairs.append(temp)

top_pairs = pd.concat(top_pairs, ignore_index=True)

plt.figure(figsize=(11, 8))

plt.barh(
    top_pairs["pair"],
    top_pairs["importance"]
)

plt.title("Most important predictors for the most predictable audio features")
plt.xlabel("Feature importance")
plt.ylabel("Prediction relationship")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
summary_table = []

for target in predictability_results.head(10)["target_feature"]:
    top_3 = (
        feature_importances[
            feature_importances["target_feature"] == target
        ]
        .sort_values("importance", ascending=False)
        .head(3)["predictor_feature"]
        .tolist()
    )
    
    r2_value = predictability_results[
        predictability_results["target_feature"] == target
    ]["r2_score"].iloc[0]
    
    summary_table.append({
        "predicted_feature": target,
        "r2_score": round(r2_value, 3),
        "top_predictors": ", ".join(top_3)
    })

summary_table = pd.DataFrame(summary_table)
summary_table

# agora fazer os perfis 
ter o csv para dar os dados do audio para ajudar
neste caso o que faço é a cada segundo vejo se o audio está normal, e houve alteração no speech ou se houve interrupções

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import ast

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

DATA_DIR = Path(".")
audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

print("Número de ficheiros áudio:", len(audio_files))

def extract_metadata_from_filename(file_path):
    filename = file_path.name.replace("_audio.pkl", "")
    candidate_part, date_part = filename.split("_vs_")
    
    candidate_1 = candidate_part.replace("_", " ")
    
    date_parts = date_part.split("_")
    month = date_parts[-2]
    day = int(date_parts[-1])
    candidate_2 = " ".join(date_parts[:-2])
    
    debate_name = f"{candidate_1} vs {candidate_2}"
    
    return {
        "source_file": file_path.name,
        "debate_name": debate_name,
        "candidate_1": candidate_1,
        "candidate_2": candidate_2,
        "month": month,
        "day": day
    }
# Se já existir o ficheiro com o mapeamento de clusters para nomes, usamos.
# Se não existir, o código usa Person 1, Person 2, Person 3.

mapping_path = Path("speaker_mapping_all_debates_named.csv")

if mapping_path.exists():
    speaker_mapping_all = pd.read_csv(mapping_path, sep=";")
    print("Mapeamento com nomes encontrado.")
else:
    speaker_mapping_all = None
    print("Mapeamento com nomes não encontrado. Vou usar Person 1, Person 2, Person 3.")


all_event_rows = []
all_profile_rows = []

for file_path in audio_files:
    print("A processar:", file_path.name)
    
    metadata = extract_metadata_from_filename(file_path)
    
    df = pd.read_pickle(file_path).copy()
    
    df["segment_start"] = df["time stamp"]
    df["segment_end"] = df["time stamp"] + df["duration"]
    
    # clustering de 3 vozes
    embeddings = np.vstack(df["speak_embeddings"].values)
    embeddings_norm = normalize(embeddings)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df["speaker_cluster"] = kmeans.fit_predict(embeddings_norm)
    
    # tentar mapear cluster -> nome real
    cluster_to_name = None
    
    if speaker_mapping_all is not None:
        row_map = speaker_mapping_all[
            speaker_mapping_all["source_file"] == file_path.name
        ]
        
        if len(row_map) > 0 and "cluster_to_name" in row_map.columns:
            try:
                cluster_to_name = ast.literal_eval(row_map.iloc[0]["cluster_to_name"])
                cluster_to_name = {int(k): v for k, v in cluster_to_name.items()}
            except:
                cluster_to_name = None
    
    # fallback: Person 1, Person 2, Person 3
    if cluster_to_name is None:
        speaker_order = (
            df.groupby("speaker_cluster")["duration"]
            .sum()
            .sort_values(ascending=False)
            .reset_index()
        )
        
        speaker_order["speaker_name"] = [f"Person {i}" for i in range(1, 4)]
        cluster_to_name = dict(
            zip(speaker_order["speaker_cluster"], speaker_order["speaker_name"])
        )
    
    df["speaker_name"] = df["speaker_cluster"].map(cluster_to_name)
    
    # calcular baseline por speaker dentro de cada debate
    speaker_baseline = (
        df.groupby("speaker_name")
        .agg(
            mean_speechrate_base=("speechrate", "mean"),
            std_speechrate_base=("speechrate", "std"),
            mean_pitchvar_base=("stdevF0Hz", "mean"),
            std_pitchvar_base=("stdevF0Hz", "std")
        )
        .reset_index()
    )
    
    df = df.merge(speaker_baseline, on="speaker_name", how="left")
    
    df["speechrate_z"] = (
        (df["speechrate"] - df["mean_speechrate_base"]) /
        df["std_speechrate_base"].replace(0, np.nan)
    )
    
    df["pitchvar_z"] = (
        (df["stdevF0Hz"] - df["mean_pitchvar_base"]) /
        df["std_pitchvar_base"].replace(0, np.nan)
    )
    
    df["speechrate_z"] = df["speechrate_z"].fillna(0)
    df["pitchvar_z"] = df["pitchvar_z"].fillna(0)
    
    max_second = int(np.ceil(df["segment_end"].max()))
    
    previous_speaker = None
    
    for sec in range(max_second + 1):
        sec_start = sec
        sec_end = sec + 1
        
        active = df[
            (df["segment_start"] < sec_end) &
            (df["segment_end"] > sec_start)
        ].copy()
        
        if active.empty:
            speaker = "No speech"
            overlap_seconds = 0
            n_active_speakers = 0
            speechrate = np.nan
            speechrate_z = np.nan
            pitchvar = np.nan
            pitchvar_z = np.nan
            segment_duration = np.nan
            npause = np.nan
        else:
            active["overlap"] = (
                np.minimum(active["segment_end"], sec_end) -
                np.maximum(active["segment_start"], sec_start)
            )
            
            # speaker principal neste segundo
            overlap_by_speaker = (
                active.groupby("speaker_name")["overlap"]
                .sum()
                .sort_values(ascending=False)
            )
            
            speaker = overlap_by_speaker.index[0]
            overlap_seconds = overlap_by_speaker.iloc[0]
            n_active_speakers = len(overlap_by_speaker)
            
            # segmento principal
            main_segment = active.sort_values("overlap", ascending=False).iloc[0]
            
            speechrate = main_segment["speechrate"]
            speechrate_z = main_segment["speechrate_z"]
            pitchvar = main_segment["stdevF0Hz"]
            pitchvar_z = main_segment["pitchvar_z"]
            segment_duration = main_segment["duration"]
            npause = main_segment["npause"]
        
        speaker_change = (
            previous_speaker is not None and
            speaker != previous_speaker and
            speaker != "No speech" and
            previous_speaker != "No speech"
        )
        
        overlap_detected = n_active_speakers > 1
        
        speechrate_increase = (
            pd.notna(speechrate_z) and speechrate_z >= 1.0
        )
        
        pitchvar_increase = (
            pd.notna(pitchvar_z) and pitchvar_z >= 1.0
        )
        
        interruption_proxy = (
            overlap_detected or speaker_change
        )
        
        visual_interest_event = (
            interruption_proxy or speechrate_increase or pitchvar_increase
        )
        
        if speaker == "No speech":
            event_type = "no_speech"
        elif interruption_proxy and speechrate_increase:
            event_type = "possible_interruption_and_speechrate_increase"
        elif interruption_proxy:
            event_type = "possible_interruption_or_speaker_change"
        elif speechrate_increase:
            event_type = "speechrate_increase"
        elif pitchvar_increase:
            event_type = "pitch_variability_increase"
        else:
            event_type = "normal_speech"
        
        all_event_rows.append({
            **metadata,
            "second": sec,
            "time_min": sec / 60,
            "speaker_name": speaker,
            "speaker_overlap_sec": overlap_seconds,
            "n_active_speakers": n_active_speakers,
            "speaker_change": int(speaker_change),
            "overlap_detected": int(overlap_detected),
            "interruption_proxy": int(interruption_proxy),
            "speechrate": speechrate,
            "speechrate_z": speechrate_z,
            "speechrate_increase": int(speechrate_increase),
            "pitch_variability": pitchvar,
            "pitchvar_z": pitchvar_z,
            "pitchvar_increase": int(pitchvar_increase),
            "segment_duration": segment_duration,
            "npause": npause,
            "visual_interest_event": int(visual_interest_event),
            "event_type": event_type
        })
        
        previous_speaker = speaker

audio_events_by_second = pd.DataFrame(all_event_rows)

audio_events_by_second.to_csv(
    "audio_events_by_second_all_debates.csv",
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

print("CSV guardado como: audio_events_by_second_all_debates.csv")
audio_events_by_second.head()

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

DATA_DIR = Path(".")
audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

print("Número de ficheiros áudio:", len(audio_files))


def extract_metadata_from_filename(file_path):
    filename = file_path.name.replace("_audio.pkl", "")
    candidate_part, date_part = filename.split("_vs_")

    candidate_1 = candidate_part.replace("_", " ")

    date_parts = date_part.split("_")
    month = date_parts[-2]
    day = int(date_parts[-1])
    candidate_2 = " ".join(date_parts[:-2])

    debate_name = f"{candidate_1} vs {candidate_2}"

    return {
        "source_file": file_path.name,
        "debate_name": debate_name,
        "candidate_1": candidate_1,
        "candidate_2": candidate_2,
        "month": month,
        "day": day
    }


# ---------------------------------------------------------
# 1. Criar clusters de voz por debate
# ---------------------------------------------------------

debate_data = {}
speaker_instances = []

for file_path in audio_files:
    metadata = extract_metadata_from_filename(file_path)

    df = pd.read_pickle(file_path).copy()
    df["segment_start"] = df["time stamp"]
    df["segment_end"] = df["time stamp"] + df["duration"]

    embeddings = np.vstack(df["speak_embeddings"].values)
    embeddings_norm = normalize(embeddings)

    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df["speaker_cluster"] = kmeans.fit_predict(embeddings_norm)

    debate_data[file_path.name] = {
        "df": df,
        "metadata": metadata,
        "embeddings_norm": embeddings_norm
    }

    for cluster in sorted(df["speaker_cluster"].unique()):
        idx = df[df["speaker_cluster"] == cluster].index
        centroid = normalize(embeddings_norm[idx].mean(axis=0).reshape(1, -1))[0]

        row = {
            **metadata,
            "speaker_cluster": cluster,
            "total_speech_sec": df.loc[idx, "duration"].sum(),
            "n_segments": len(idx)
        }

        for i, value in enumerate(centroid):
            row[f"emb_{i:03d}"] = value

        speaker_instances.append(row)

speaker_instances = pd.DataFrame(speaker_instances)

emb_cols = [col for col in speaker_instances.columns if col.startswith("emb_")]

print("Speaker instances criadas:", speaker_instances.shape)


# ---------------------------------------------------------
# 2. Criar embedding provável para cada candidato
# ---------------------------------------------------------

candidates = sorted(
    set(speaker_instances["candidate_1"]).union(set(speaker_instances["candidate_2"]))
)

candidate_embeddings = {}

for candidate in candidates:
    candidate_instances = speaker_instances[
        (speaker_instances["candidate_1"] == candidate) |
        (speaker_instances["candidate_2"] == candidate)
    ].copy()

    debate_names = candidate_instances["debate_name"].unique()

    best_score = -999
    best_prototype = None

    for seed_idx in candidate_instances.index:
        prototype = candidate_instances.loc[seed_idx, emb_cols].values.astype(float)
        prototype = normalize(prototype.reshape(1, -1))[0]

        for _ in range(10):
            selected_embeddings = []

            for debate in debate_names:
                group = candidate_instances[candidate_instances["debate_name"] == debate]
                group_embs = group[emb_cols].values.astype(float)

                sims = group_embs @ prototype
                best_emb = group_embs[np.argmax(sims)]
                selected_embeddings.append(best_emb)

            selected_embeddings = np.vstack(selected_embeddings)
            new_prototype = normalize(selected_embeddings.mean(axis=0).reshape(1, -1))[0]

            if np.allclose(prototype, new_prototype, atol=1e-5):
                break

            prototype = new_prototype

        sims_final = selected_embeddings @ prototype
        score = sims_final.mean()

        if score > best_score:
            best_score = score
            best_prototype = prototype

    candidate_embeddings[candidate] = {
        "embedding": best_prototype,
        "score": best_score
    }

candidate_embedding_quality = pd.DataFrame([
    {
        "candidate": candidate,
        "embedding_score": values["score"]
    }
    for candidate, values in candidate_embeddings.items()
]).round(3)

candidate_embedding_quality